# 04 - Main Backtest

**Purpose.** Drive `backtest.run_baselines.run_all` and produce the publication-quality equity-curve figure plus drawdown plot with rebalance markers.

**Prerequisites.** `fractal-defi==1.3.2` available; baselines / Predictive MCDM importable. The trained ONNX is optional; the Predictive MCDM strategy is skipped if missing.

**Expected runtime.** 1-3 minutes on the synthetic panel.


In [ ]:
# --- path preamble: make sibling packages importable ---
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "extras" / "fractal_pr_lending_allocation"))

print(f"ROOT = {ROOT}")


In [ ]:
# Synthetic-data fallback: mirrors forecaster.train._make_synth_df.
# Used whenever the real joined_clean.parquet is not yet on disk.
import numpy as np
import pandas as pd


def make_synth_joined(n_rows: int = 2000, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2025-01-01", periods=n_rows, freq="h", tz="UTC")

    def util_walk(start: float) -> np.ndarray:
        u = np.empty(n_rows)
        u[0] = start
        for i in range(1, n_rows):
            u[i] = np.clip(u[i - 1] + rng.normal(0.0, 0.01), 0.05, 0.97)
        return u

    u_a = util_walk(0.55)
    u_c = util_walk(0.45)

    # Toy rate process: kink-shaped baseline plus Gaussian residual.
    r_a = 0.05 * (u_a / 0.92) * 0.90 * u_a + rng.normal(0, 0.002, n_rows)
    r_c = 0.04 * u_c + rng.normal(0, 0.002, n_rows)
    r_a = np.clip(r_a, 0.0, 0.5)
    r_c = np.clip(r_c, 0.0, 0.5)

    tvl_a = 1e8 + np.cumsum(rng.normal(0, 1e5, n_rows))
    tvl_c = 5e7 + np.cumsum(rng.normal(0, 5e4, n_rows))
    gas = np.clip(
        20 + 5 * rng.standard_normal(n_rows) + 10 * np.sin(np.arange(n_rows) / 24),
        5, 200,
    )
    eth = 3000 + np.cumsum(rng.normal(0, 5, n_rows))

    return pd.DataFrame({
        "r_aave": r_a, "r_compound": r_c,
        "u_aave": u_a, "u_compound": u_c,
        "tvl_aave": tvl_a, "tvl_compound": tvl_c,
        "gas_gwei": gas, "eth_usd": eth,
    }, index=idx)


def load_joined(path: str = "data/cached/joined_clean.parquet") -> tuple[pd.DataFrame, bool]:
    """Try the real cached panel; fall back to synthetic on FileNotFoundError."""
    full = ROOT / path
    try:
        df = pd.read_parquet(full)
        print(f"[real] loaded {len(df):,} rows from {full}")
        return df, True
    except FileNotFoundError:
        print(f"[synth] {full} not found - generating synthetic panel")
        return make_synth_joined(), False


df, is_real = load_joined()
df.head()


## 1. Run all four strategies (synthetic panel)

In [ ]:
from backtest.run_baselines import run_all, BaselineRunConfig

cfg = BaselineRunConfig(initial_balance=1_000_000.0)
results_df = run_all(cfg, synthetic=True)
results_df


## 2. Load the summary CSV

In [ ]:
import pandas as pd

csv_path = ROOT / 'results' / 'tables' / 'baselines.csv'
try:
    summary = pd.read_csv(csv_path)
    print(f'[loaded] {csv_path}')
    print(summary.to_string(index=False))
except FileNotFoundError:
    print(f'[warn] {csv_path} not found - did run_all() succeed?')
    summary = pd.DataFrame()
summary


## 3. Equity curves with annotations

In [ ]:
import matplotlib.pyplot as plt
from pathlib import Path

fig_path = ROOT / 'results' / 'figures' / 'baselines_equity.png'
if fig_path.exists():
    from PIL import Image
    img = Image.open(fig_path)
    fig, ax = plt.subplots(figsize=(11, 6))
    ax.imshow(img); ax.axis('off')
    ax.set_title(f'Equity-curve overlay (from {fig_path.name})')
    plt.tight_layout(); plt.show()
else:
    print('Equity-curve figure not yet generated; re-run run_all() above.')


## 4. Drawdown plot from summary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

if not summary.empty and 'max_dd' in summary.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(summary['strategy'], summary['max_dd'] * 100, color='C3', alpha=0.75)
    ax.set_ylabel('max drawdown (%)')
    ax.set_title('Maximum drawdown by strategy (test window)')
    ax.tick_params(axis='x', rotation=20)
    for i, v in enumerate(summary['max_dd'] * 100):
        ax.text(i, v, f'{v:.2f}%', ha='center', va='bottom')
    plt.tight_layout(); plt.show()
else:
    print('Summary frame empty or missing max_dd column.')


## 5. Rebalance counts

In [ ]:
if not summary.empty and 'n_rebalances' in summary.columns:
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(summary['strategy'], summary['n_rebalances'], color='C2', alpha=0.8)
    ax.set_ylabel('# rebalances over test window')
    ax.set_title('Strategy turnover (proxy: balance-flip count)')
    ax.tick_params(axis='x', rotation=20)
    plt.tight_layout(); plt.show()
else:
    print('Summary frame missing n_rebalances column.')

# Caption: Rebalance turnover. The plan's anti-pathology constraint:
# Predictive MCDM turnover < 2x the lowest baseline turnover.


## Next steps

- Move to `05_ablations_forecast_value.ipynb` to compare forecaster variants.
- The final whitepaper headline figure is the equity-curve overlay above.

Relevant plan section: **PROJECT_2_PLAN.md S6 (Backtesting Protocol), S7 (Baselines), S9 (Metrics).**
